
# Vietnamese Sign Language Recognition for the Hearing Impaired

This notebook **fully replaces the unrelated MT baseline** and builds an end-to-end pipeline for the task shown in the provided brief:

- read the dataset in the official folder structure,
- train a video classifier on `train/<class_name>/*.mp4`,
- validate with **Macro-F1**,
- run inference on `test/*.mp4`,
- export the official `video_name,label` CSV,
- compress the CSV into a `.zip` submission.

The implementation below is designed to be practical and adaptable:

- **frame encoder:** ImageNet-style 2D CNN backbone,
- **temporal encoder:** BiGRU over sampled video frames,
- **pooling:** attention pooling over time,
- **training:** AdamW + AMP + gradient clipping + early stopping,
- **inference:** supports checkpoint ensembling across folds.

---

## Faithful English rendering of the challenge brief

### 1. Background
In daily communication, hearing-impaired people often use sign language to express information. However, the biggest barrier is that a large part of the community does not understand it, which leads to communication difficulties. With the development of Computer Vision and Deep Learning, building an automatic sign language recognition system has become feasible. In this contest, participants are asked to build a sign language recognition system to support hearing-impaired people in communication and access to information. The system must be able to classify sign-language videos into the corresponding words or phrases.

### 2. Problem description
The dataset consists of videos showing hand signs corresponding to words or phrases in Vietnamese sign language. The task is to build a model that recognizes the corresponding sign. Concretely, the original brief states the input as:

\[
X = \{I_1, I_2, \ldots, I_n\}
\]

where each \(I_i\) is an image of a hand performing a sign. The goal is to predict the label \(y_i\) corresponding to the word or phrase represented by the hand sign:

\[
\hat{y}_i = f_\theta(I_i)
\]

where \(f_\theta\) is a trained deep model.

> Note: the brief writes the math in image form, but the provided dataset format is **video classification**, so this notebook treats each sample as a video / image sequence.

### 3. Data and baseline model
- **Training data:** short videos containing hand signs, with corresponding labels (for example: `An ủi`, `Xin lỗi`, `Cảm ơn`, ...).
- **Test data:** unlabeled videos used for model evaluation.
- Participants are provided with a pretrained baseline described in the brief as an **"imagenet CRNN"** and also as a **3D-CNN architecture** for video action recognition.
- The data were captured under varying lighting conditions, camera angles, performers, and skin tones to ensure diversity.

### 4. Requirements
- Train an image / image-sequence classification model that accurately recognizes the hand sign.
- The output must be the label of the word or phrase corresponding to the input video.
- Participants are encouraged to add preprocessing steps or improve the model to increase accuracy.

### 5. Evaluation metric
The overall metric is **Macro-F1**, i.e. the average F1 score across all classes.

For each class \(i\):

\[
\text{Precision}_i = \frac{TP_i}{TP_i + FP_i}
\]

\[
\text{Recall}_i = \frac{TP_i}{TP_i + FN_i}
\]

\[
F1_i = 2 \times \frac{\text{Precision}_i \times \text{Recall}_i}{\text{Precision}_i + \text{Recall}_i}
\]

and

\[
\text{Macro-F1} = \frac{1}{N}\sum_{i=1}^{N} F1_i
\]

where \(N\) is the number of classes.

### 6. Dataset and submission format
Official structure:

```text
dataset/
│
├── train/
│   ├── An ủi/
│   │   ├── xxx.mp4
│   │   └── ...
│   ├── Bạn ngày/
│   │   └── ...
│   └── ...
│
├── test/
│   ├── xxx.mp4
│   └── ...
│
└── label_mapping.pkl
```

- `train/` contains the training videos, split into class folders.
- the **folder name is the class name**,
- each class folder contains `.mp4` videos,
- `label_mapping.pkl` maps class names and internal class ids,
- `test/` contains unlabeled videos.

Required submission:

```csv
video_name,label
239890_2.mp4,Xa
794299_2.mp4,Thức ăn
362778.mp4,Đi
681343.mp4,Chạy
```

Then compress the CSV file into a `.zip` before submission.


In [ ]:

# Uncomment if your environment is missing dependencies.
# !pip install -q opencv-python scikit-learn pandas matplotlib tqdm


In [ ]:

import os
import cv2
import math
import time
import copy
import json
import random
import zipfile
import pickle
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)
print("torch:", torch.__version__)


## Configuration

In [ ]:

# =========================
# Paths
# =========================
DATA_ROOT = Path("./dataset")            # change this to your dataset root
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
LABEL_MAPPING_PATH = DATA_ROOT / "label_mapping.pkl"
WORK_DIR = Path("./workdir_sign")
WORK_DIR.mkdir(parents=True, exist_ok=True)

# =========================
# Reproducibility
# =========================
SEED = 42

# =========================
# Data
# =========================
NUM_FRAMES = 16
IMG_SIZE = 224
NUM_WORKERS = 2
PIN_MEMORY = True
USE_WEIGHTED_SAMPLER = False

# =========================
# CV / training
# =========================
N_SPLITS = 5             # set to 1 if you want a single train/valid split workflow
FOLDS_TO_RUN = [0]       # example: [0, 1, 2, 3, 4] for full CV
EPOCHS = 12
BATCH_SIZE = 4
LR = 2e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0
EARLY_STOPPING = 4
USE_AMP = True

# =========================
# Model
# =========================
BACKBONE = "resnet18"   # practical, stable baseline
PRETRAINED = True        # falls back gracefully if pretrained weights are unavailable
GRU_HIDDEN = 256
GRU_LAYERS = 1
DROPOUT = 0.30

# =========================
# Inference / export
# =========================
SUBMISSION_CSV = WORK_DIR / "submission.csv"
SUBMISSION_ZIP = WORK_DIR / "submission.zip"


In [ ]:

def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)


## Dataset discovery and label mapping

In [ ]:

def _normalize_mapping_obj(obj):
    """
    Robustly parse possible label_mapping.pkl layouts.
    Returns: name_to_id, id_to_name
    """
    if isinstance(obj, dict):
        # case 1: {class_name: class_id}
        if all(isinstance(k, str) for k in obj.keys()):
            name_to_id = {str(k): int(v) for k, v in obj.items()}
            id_to_name = {v: k for k, v in name_to_id.items()}
            return name_to_id, id_to_name

        # case 2: {class_id: class_name}
        if all(isinstance(k, (int, np.integer)) for k in obj.keys()):
            id_to_name = {int(k): str(v) for k, v in obj.items()}
            name_to_id = {v: k for k, v in id_to_name.items()}
            return name_to_id, id_to_name

        # case 3: nested dictionaries
        for key_name in ["name_to_id", "class_to_id", "label_to_id"]:
            if key_name in obj and isinstance(obj[key_name], dict):
                name_to_id = {str(k): int(v) for k, v in obj[key_name].items()}
                id_to_name = {v: k for k, v in name_to_id.items()}
                return name_to_id, id_to_name
        for key_name in ["id_to_name", "id_to_class", "id_to_label"]:
            if key_name in obj and isinstance(obj[key_name], dict):
                id_to_name = {int(k): str(v) for k, v in obj[key_name].items()}
                name_to_id = {v: k for k, v in id_to_name.items()}
                return name_to_id, id_to_name

    raise ValueError("Unsupported label_mapping.pkl format")


def load_label_mapping(train_dir: Path, label_mapping_path: Path | None = None):
    if label_mapping_path is not None and label_mapping_path.exists():
        with open(label_mapping_path, "rb") as f:
            raw = pickle.load(f)
        try:
            return _normalize_mapping_obj(raw)
        except Exception as exc:
            print(f"Could not parse label_mapping.pkl directly ({exc}). Falling back to folder scan.")

    class_names = sorted([p.name for p in train_dir.iterdir() if p.is_dir()])
    name_to_id = {name: idx for idx, name in enumerate(class_names)}
    id_to_name = {idx: name for name, idx in name_to_id.items()}
    return name_to_id, id_to_name


def build_train_dataframe(train_dir: Path, name_to_id: dict):
    rows = []
    for class_dir in sorted([p for p in train_dir.iterdir() if p.is_dir()]):
        class_name = class_dir.name
        label_id = name_to_id[class_name]
        for video_path in sorted(class_dir.glob("*.mp4")):
            rows.append({
                "video_path": str(video_path),
                "video_name": video_path.name,
                "class_name": class_name,
                "label_id": label_id,
            })
    return pd.DataFrame(rows)


def build_test_dataframe(test_dir: Path):
    rows = []
    for video_path in sorted(test_dir.glob("*.mp4")):
        rows.append({
            "video_path": str(video_path),
            "video_name": video_path.name,
        })
    return pd.DataFrame(rows)


In [ ]:

assert TRAIN_DIR.exists(), f"Train directory not found: {TRAIN_DIR}"
assert TEST_DIR.exists(), f"Test directory not found: {TEST_DIR}"

name_to_id, id_to_name = load_label_mapping(TRAIN_DIR, LABEL_MAPPING_PATH)
train_df = build_train_dataframe(TRAIN_DIR, name_to_id)
test_df = build_test_dataframe(TEST_DIR)

print("num train videos:", len(train_df))
print("num test videos :", len(test_df))
print("num classes     :", len(name_to_id))
print("sample classes  :", list(name_to_id.keys())[:10])

display(train_df.head())
display(test_df.head())


In [ ]:

# Basic dataset sanity checks
assert train_df["class_name"].nunique() == len(name_to_id)
assert train_df["label_id"].nunique() == len(name_to_id)
assert train_df["video_name"].is_unique, "Expected unique video names in train set"
assert test_df["video_name"].is_unique, "Expected unique video names in test set"

class_counts = train_df["class_name"].value_counts().sort_values(ascending=False)
print(class_counts.describe())

plt.figure(figsize=(14, 4))
class_counts.plot(kind="bar")
plt.title("Training videos per class")
plt.xlabel("Class")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## Video loading and frame sampling

In [ ]:

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)


def _uniform_indices(num_available: int, num_frames: int):
    if num_available <= 0:
        return [0] * num_frames
    if num_available >= num_frames:
        return np.linspace(0, num_available - 1, num_frames).astype(int).tolist()
    # repeat frames if the clip is shorter than requested
    idx = np.linspace(0, num_available - 1, num_frames)
    return np.clip(np.round(idx), 0, num_available - 1).astype(int).tolist()


def _segment_jitter_indices(num_available: int, num_frames: int):
    if num_available <= 0:
        return [0] * num_frames
    if num_available < num_frames:
        return _uniform_indices(num_available, num_frames)

    boundaries = np.linspace(0, num_available, num_frames + 1).astype(int)
    indices = []
    for i in range(num_frames):
        start = boundaries[i]
        end = max(boundaries[i + 1], start + 1)
        indices.append(np.random.randint(start, end))
    return indices


def read_video_frames(video_path, num_frames=16, image_size=224, train_mode=False):
    """
    Returns tensor of shape [T, C, H, W].
    """
    cap = cv2.VideoCapture(str(video_path))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        frames = []
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frames.append(frame)
        cap.release()
        total_frames = len(frames)
        if total_frames == 0:
            raise RuntimeError(f"Could not decode any frame from: {video_path}")
        indices = _segment_jitter_indices(total_frames, num_frames) if train_mode else _uniform_indices(total_frames, num_frames)
        selected = [frames[i] for i in indices]
    else:
        indices = _segment_jitter_indices(total_frames, num_frames) if train_mode else _uniform_indices(total_frames, num_frames)
        selected = []
        for idx in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
            ok, frame = cap.read()
            if not ok:
                # fallback to last valid frame if possible
                if selected:
                    frame = selected[-1]
                else:
                    cap.release()
                    raise RuntimeError(f"Failed to read frame {idx} from: {video_path}")
            selected.append(frame)
        cap.release()

    tensor_frames = []
    for frame in selected:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (image_size, image_size), interpolation=cv2.INTER_LINEAR)
        frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0
        tensor_frames.append(frame)
    frames = torch.stack(tensor_frames, dim=0)  # [T, C, H, W]
    return frames


def sequence_augment(frames: torch.Tensor):
    """
    Apply light sequence-consistent augmentation.
    frames: [T, C, H, W] in [0, 1]
    """
    if random.random() < 0.5:
        brightness = random.uniform(0.90, 1.10)
        frames = torch.clamp(frames * brightness, 0.0, 1.0)

    if random.random() < 0.5:
        contrast = random.uniform(0.90, 1.10)
        mean = frames.mean(dim=(-1, -2), keepdim=True)
        frames = torch.clamp((frames - mean) * contrast + mean, 0.0, 1.0)

    if random.random() < 0.2:
        noise = torch.randn_like(frames) * 0.01
        frames = torch.clamp(frames + noise, 0.0, 1.0)

    return frames


def normalize_frames(frames: torch.Tensor):
    return (frames - IMAGENET_MEAN) / IMAGENET_STD


In [ ]:

# Quick visual sanity check on one training video
sample_video_path = Path(train_df.iloc[0]["video_path"])
sample_frames = read_video_frames(sample_video_path, num_frames=8, image_size=IMG_SIZE, train_mode=False)
print("sample video:", sample_video_path.name)
print("tensor shape:", tuple(sample_frames.shape))

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, frame in zip(axes.flatten(), sample_frames):
    ax.imshow(frame.permute(1, 2, 0).numpy())
    ax.axis("off")
plt.tight_layout()
plt.show()


## PyTorch dataset and dataloaders

In [ ]:

class SignVideoDataset(Dataset):
    def __init__(self, df: pd.DataFrame, num_frames=16, image_size=224, train_mode=False, labeled=True):
        self.df = df.reset_index(drop=True).copy()
        self.num_frames = num_frames
        self.image_size = image_size
        self.train_mode = train_mode
        self.labeled = labeled

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = row["video_path"]
        frames = read_video_frames(
            video_path=video_path,
            num_frames=self.num_frames,
            image_size=self.image_size,
            train_mode=self.train_mode,
        )
        if self.train_mode:
            frames = sequence_augment(frames)
        frames = normalize_frames(frames)

        sample = {
            "frames": frames,                    # [T, C, H, W]
            "video_name": row["video_name"],
            "video_path": row["video_path"],
        }
        if self.labeled:
            sample["label"] = int(row["label_id"])
            sample["class_name"] = row["class_name"]
        return sample


def make_loader(df, train_mode, labeled=True, sampler=None, shuffle=None):
    ds = SignVideoDataset(
        df=df,
        num_frames=NUM_FRAMES,
        image_size=IMG_SIZE,
        train_mode=train_mode,
        labeled=labeled,
    )
    if shuffle is None:
        shuffle = train_mode and sampler is None
    return DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )


def build_weighted_sampler(labels):
    class_counts = Counter(labels)
    weights = [1.0 / class_counts[int(label)] for label in labels]
    return WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)


## Model: CNN frame encoder + BiGRU + temporal attention

In [ ]:

def build_cnn_backbone(name="resnet18", pretrained=True):
    if name != "resnet18":
        raise ValueError(f"Unsupported backbone: {name}")

    weights = None
    if pretrained:
        try:
            weights = models.ResNet18_Weights.DEFAULT
        except Exception:
            weights = None

    try:
        backbone = models.resnet18(weights=weights)
    except Exception:
        backbone = models.resnet18(weights=None)

    out_dim = backbone.fc.in_features
    backbone.fc = nn.Identity()
    return backbone, out_dim


class TemporalAttentionPooling(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        # x: [B, T, D]
        scores = self.attn(x).squeeze(-1)        # [B, T]
        weights = torch.softmax(scores, dim=1)   # [B, T]
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class SignLanguageRecognizer(nn.Module):
    def __init__(self, num_classes, backbone_name="resnet18", pretrained=True,
                 gru_hidden=256, gru_layers=1, dropout=0.3):
        super().__init__()
        self.backbone, feat_dim = build_cnn_backbone(backbone_name, pretrained=pretrained)
        self.gru = nn.GRU(
            input_size=feat_dim,
            hidden_size=gru_hidden,
            num_layers=gru_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if gru_layers > 1 else 0.0,
        )
        self.pool = TemporalAttentionPooling(in_dim=gru_hidden * 2, hidden_dim=gru_hidden)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(gru_hidden * 2, num_classes)

    def forward(self, frames):
        # frames: [B, T, C, H, W]
        b, t, c, h, w = frames.shape
        x = frames.view(b * t, c, h, w)
        feats = self.backbone(x)                  # [B*T, D]
        feats = feats.view(b, t, -1)             # [B, T, D]
        seq, _ = self.gru(feats)                 # [B, T, 2H]
        pooled, attn = self.pool(seq)            # [B, 2H]
        logits = self.head(self.dropout(pooled))
        return logits, attn


In [ ]:

# Smoke test
model = SignLanguageRecognizer(
    num_classes=len(name_to_id),
    backbone_name=BACKBONE,
    pretrained=False,
    gru_hidden=GRU_HIDDEN,
    gru_layers=GRU_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

x = torch.randn(2, NUM_FRAMES, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
with torch.no_grad():
    logits, attn = model(x)
print("logits shape:", tuple(logits.shape))
print("attn shape  :", tuple(attn.shape))


## Training utilities

In [ ]:

def macro_f1_from_logits(logits, targets):
    preds = logits.argmax(dim=1).detach().cpu().numpy()
    targets = targets.detach().cpu().numpy()
    return f1_score(targets, preds, average="macro")


class AverageMeter:
    def __init__(self):
        self.reset()

    def reset(self):
        self.sum = 0.0
        self.count = 0

    @property
    def avg(self):
        return self.sum / max(self.count, 1)

    def update(self, value, n=1):
        self.sum += value * n
        self.count += n


def train_one_epoch(model, loader, optimizer, criterion, scaler=None, scheduler=None):
    model.train()
    loss_meter = AverageMeter()
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc="train", leave=False):
        frames = batch["frames"].to(DEVICE, non_blocking=True)
        targets = batch["label"].to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
            logits, _ = model(frames)
            loss = criterion(logits, targets)

        if scaler is not None and USE_AMP and DEVICE.type == "cuda":
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        loss_meter.update(loss.item(), frames.size(0))
        all_preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_targets.extend(targets.detach().cpu().tolist())

    epoch_f1 = f1_score(all_targets, all_preds, average="macro")
    return loss_meter.avg, epoch_f1


@torch.no_grad()
def valid_one_epoch(model, loader, criterion):
    model.eval()
    loss_meter = AverageMeter()
    all_preds, all_targets = [], []

    for batch in tqdm(loader, desc="valid", leave=False):
        frames = batch["frames"].to(DEVICE, non_blocking=True)
        targets = batch["label"].to(DEVICE, non_blocking=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP and DEVICE.type == "cuda"):
            logits, _ = model(frames)
            loss = criterion(logits, targets)

        loss_meter.update(loss.item(), frames.size(0))
        all_preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_targets.extend(targets.detach().cpu().tolist())

    epoch_f1 = f1_score(all_targets, all_preds, average="macro")
    return loss_meter.avg, epoch_f1, all_targets, all_preds


In [ ]:

def fit_one_fold(train_fold_df, valid_fold_df, fold_idx=0):
    train_sampler = None
    if USE_WEIGHTED_SAMPLER:
        train_sampler = build_weighted_sampler(train_fold_df["label_id"].tolist())

    train_loader = make_loader(train_fold_df, train_mode=True, labeled=True, sampler=train_sampler)
    valid_loader = make_loader(valid_fold_df, train_mode=False, labeled=True)

    model = SignLanguageRecognizer(
        num_classes=len(name_to_id),
        backbone_name=BACKBONE,
        pretrained=PRETRAINED,
        gru_hidden=GRU_HIDDEN,
        gru_layers=GRU_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and DEVICE.type == "cuda")

    best_f1 = -1.0
    best_state = None
    history = []
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, scaler=scaler, scheduler=None)
        valid_loss, valid_f1, y_true, y_pred = valid_one_epoch(model, valid_loader, criterion)
        scheduler.step()
        elapsed = time.time() - start

        row = {
            "fold": fold_idx,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_f1": train_f1,
            "valid_loss": valid_loss,
            "valid_f1": valid_f1,
            "minutes": elapsed / 60.0,
        }
        history.append(row)
        print(row)

        if valid_f1 > best_f1:
            best_f1 = valid_f1
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, WORK_DIR / f"best_fold{fold_idx}.pt")
            patience = 0
        else:
            patience += 1
            if patience >= EARLY_STOPPING:
                print(f"Early stopping on fold {fold_idx} at epoch {epoch}")
                break

    history_df = pd.DataFrame(history)
    model.load_state_dict(best_state)
    valid_loss, valid_f1, y_true, y_pred = valid_one_epoch(model, valid_loader, criterion)

    report = classification_report(
        y_true,
        y_pred,
        target_names=[id_to_name[i] for i in range(len(id_to_name))],
        zero_division=0,
        output_dict=True,
    )
    report_df = pd.DataFrame(report).T
    report_df.to_csv(WORK_DIR / f"classification_report_fold{fold_idx}.csv")
    history_df.to_csv(WORK_DIR / f"history_fold{fold_idx}.csv", index=False)

    return {
        "model": model,
        "history_df": history_df,
        "report_df": report_df,
        "best_f1": valid_f1,
        "y_true": y_true,
        "y_pred": y_pred,
    }


## Fold split and training

In [ ]:

if N_SPLITS <= 1:
    # fallback single split behavior
    from sklearn.model_selection import train_test_split
    train_idx, valid_idx = train_test_split(
        np.arange(len(train_df)),
        test_size=0.2,
        random_state=SEED,
        stratify=train_df["label_id"].values,
    )
    folds = [(train_idx, valid_idx)]
else:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    folds = list(skf.split(train_df, train_df["label_id"]))

print(f"prepared {len(folds)} folds")


In [ ]:

all_fold_results = []

for fold_idx, (train_idx, valid_idx) in enumerate(folds):
    if fold_idx not in FOLDS_TO_RUN:
        continue

    print("=" * 80)
    print(f"FOLD {fold_idx}")
    print("=" * 80)

    train_fold_df = train_df.iloc[train_idx].reset_index(drop=True)
    valid_fold_df = train_df.iloc[valid_idx].reset_index(drop=True)

    result = fit_one_fold(train_fold_df, valid_fold_df, fold_idx=fold_idx)
    result["fold"] = fold_idx
    all_fold_results.append(result)

print("finished folds:", [r["fold"] for r in all_fold_results])
if all_fold_results:
    print("mean CV Macro-F1:", np.mean([r["best_f1"] for r in all_fold_results]))


In [ ]:

# Optional: inspect one fold history
if all_fold_results:
    display(all_fold_results[0]["history_df"])

    plt.figure(figsize=(8, 4))
    plt.plot(all_fold_results[0]["history_df"]["epoch"], all_fold_results[0]["history_df"]["train_f1"], label="train_f1")
    plt.plot(all_fold_results[0]["history_df"]["epoch"], all_fold_results[0]["history_df"]["valid_f1"], label="valid_f1")
    plt.xlabel("Epoch")
    plt.ylabel("Macro-F1")
    plt.title("Fold 0 learning curve")
    plt.legend()
    plt.tight_layout()
    plt.show()


## Inference and checkpoint ensembling

In [ ]:

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    logits_list = []
    video_names = []

    for batch in tqdm(loader, desc="predict", leave=False):
        frames = batch["frames"].to(DEVICE, non_blocking=True)
        logits, _ = model(frames)
        logits_list.append(logits.detach().cpu())
        video_names.extend(batch["video_name"])

    return torch.cat(logits_list, dim=0), video_names


def load_fold_model(checkpoint_path: Path):
    model = SignLanguageRecognizer(
        num_classes=len(name_to_id),
        backbone_name=BACKBONE,
        pretrained=False,
        gru_hidden=GRU_HIDDEN,
        gru_layers=GRU_LAYERS,
        dropout=DROPOUT,
    ).to(DEVICE)
    state = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(state)
    model.eval()
    return model


In [ ]:

test_loader = make_loader(test_df, train_mode=False, labeled=False)
checkpoint_paths = sorted(WORK_DIR.glob("best_fold*.pt"))
assert checkpoint_paths, f"No checkpoints found in {WORK_DIR}. Train at least one fold first."
print("checkpoints:", checkpoint_paths)

all_test_logits = None
final_video_names = None

for ckpt in checkpoint_paths:
    model = load_fold_model(ckpt)
    logits, video_names = predict_logits(model, test_loader)

    if all_test_logits is None:
        all_test_logits = logits
        final_video_names = video_names
    else:
        all_test_logits += logits

all_test_logits = all_test_logits / len(checkpoint_paths)
test_pred_ids = all_test_logits.argmax(dim=1).numpy()
test_pred_names = [id_to_name[int(idx)] for idx in test_pred_ids]

submission_df = pd.DataFrame({
    "video_name": final_video_names,
    "label": test_pred_names,
})
submission_df.head()


## Export official submission CSV and ZIP

In [ ]:

submission_df.to_csv(SUBMISSION_CSV, index=False, encoding="utf-8-sig")
print(f"saved CSV: {SUBMISSION_CSV}")

with zipfile.ZipFile(SUBMISSION_ZIP, mode="w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(SUBMISSION_CSV, arcname=SUBMISSION_CSV.name)

print(f"saved ZIP: {SUBMISSION_ZIP}")
submission_df.head()



## Practical improvement ideas

If you want to push performance further, the next upgrades that usually matter most for this task are:

1. **More temporal coverage**: increase `NUM_FRAMES` from 16 to 24/32 if GPU memory allows.
2. **Cross-validation ensembling**: train all folds and average logits.
3. **Stronger frame encoder**: switch from `resnet18` to a stronger pretrained backbone.
4. **Better augmentations**: hand-safe augmentations only — avoid random horizontal flip unless you are certain mirroring does not change sign meaning.
5. **Sequence models**: replace BiGRU with temporal transformer / TCN / 3D CNN if the motion pattern is especially important.
6. **Class balancing**: turn on `USE_WEIGHTED_SAMPLER` if the per-class counts are imbalanced.

This notebook is intentionally written so these changes can be made with minimal code edits.
